In [ ]:
pip install langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Runnable 고급 — 병렬 처리와 조건 분기

In [ ]:
# chain = prompt | llm | parser
# LCEL : LangCahin Expressing Language

In [ ]:
# RunnableSequence
# RunnableParallel
# RunnablePassthrough

### LangChain 핵심 클래스 요약

1. **RunnableSequence (연결)**: 단계를 차례대로 실행함. (`A | B | C`)
2. **RunnableParallel (병렬)**: 여러 작업을 동시에 수행하고 결과를 딕셔너리로 묶음.
3. **RunnablePassthrough (전달)**: 입력을 수정 없이 그대로 다음 단계로 넘겨줌.
4. **RunnableBranch (분기)**: 조건에 따라 실행할 경로를 선택함.

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
  summary = ChatPromptTemplate.from_template('{text}를 한 줄로 요약해주세요.'),
  keywords = ChatPromptTemplate.from_template('{text}에서 키워드를 3개만 뽑아주세요.')
)


In [ ]:
result = parallel_chain.invoke({"text": "Langchain은 LLM 기반 application 개발 프레임워크입니다."})

In [ ]:
result

{'summary': ChatPromptValue(messages=[HumanMessage(content='Langchain은 LLM 기반 application 개발 프레임워크입니다.를 한 줄로 요약해주세요.', additional_kwargs={}, response_metadata={})]),
 'keywords': ChatPromptValue(messages=[HumanMessage(content='Langchain은 LLM 기반 application 개발 프레임워크입니다.에서 키워드를 3개만 뽑아주세요.', additional_kwargs={}, response_metadata={})])}

In [ ]:
llm = ChatOpenAI(model = "gpt-4o-mini")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
import os
from google.colab import userdata

# Colab 비밀번호 메뉴에 'OPENAI_API_KEY'가 저장되어 있어야 합니다.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
print("LLM이 성공적으로 초기화되었습니다.")

LLM이 성공적으로 초기화되었습니다.


In [ ]:
# RunnableBranch: 입력에 따라 서로 다른 실행 경로로 분기를 만드는 클래스입니다.

from langchain_core.runnables import RunnableBranch

tech_chain = ChatPromptTemplate.from_template("기술지원팀입니다 : {question}")
billing_chain = ChatPromptTemplate.from_template("요금 관리 팀입니다 : {question}")
general_chain = ChatPromptTemplate.from_template("일반 상담 팀입니다 : {question}")

def route_logic(x):
  text = x['question']
  if ('오류' in text) or ('에러' in text):
    return 'technical'
  elif '가격' in text or '요금' in text:
    return 'billing'
  else:
    return 'general'

branch = RunnableBranch(
    (lambda x : route_logic(x) == 'technical', tech_chain),# x값을 함수에 넣은 값이 테크니컬이라면 테크 체인을 작동시키라~
    (lambda x : route_logic(x) == 'billing', billing_chain),
    general_chain

)

questions = ["프린터 오류가 났습니다.", "월 요금이 얼마인가요?", "영업시간 알려주세요"]
for q in questions:
  print(f"Q:{q}\n A: {branch.invoke({"question":q})}\n")

Q:프린터 오류가 났습니다.
 A: messages=[HumanMessage(content='기술지원팀입니다 : 프린터 오류가 났습니다.', additional_kwargs={}, response_metadata={})]

Q:월 요금이 얼마인가요?
 A: messages=[HumanMessage(content='요금 관리 팀입니다 : 월 요금이 얼마인가요?', additional_kwargs={}, response_metadata={})]

Q:영업시간 알려주세요
 A: messages=[HumanMessage(content='일반 상담 팀입니다 : 영업시간 알려주세요', additional_kwargs={}, response_metadata={})]



In [ ]:
SDAzsdrftdsadrtf

### lambda (익명 함수)란?
`lambda` 키워드는 별도의 `def` 예약어를 사용해 함수를 정의하지 않고, 한 줄로 간단하게 함수를 만들 때 사용함.

**기본 구조:**
`lambda 인자 : 표현식`

In [ ]:
# 1. 일반적인 함수 정의
def add_one(x):
    return x + 1

# 2. lambda를 이용한 동일한 기능 정의
lambda_add_one = lambda x : x + 1

print(f"일반 함수 결과: {add_one(5)}")
print(f"lambda 결과: {lambda_add_one(5)}")

# LangChain의 RunnableBranch에서는 다음과 같이 활용됩니다:
# (lambda x : route_logic(x) == 'technical', tech_chain)
# 이는 '입력 x를 받아서 route_logic(x) 결과가 technical인지 확인하는 함수'를 즉석에서 만든 것입니다.

## Gradio로 챗봇 UI 구성

# 그라디오(Gradio) 라이브러리

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
def greet(name):
  return f"안녕하세요 {name}님"

demo1 = gr.Interface(
    fn = greet,  # fnction은 입력에 대한 인터페이스가 어떻게 반응하는지
    inputs = gr.Textbox(label='이름입력'), # 입력 값을 어떻게 받을것인지
    outputs = gr.Textbox(label='인사말'),
    title = '인사봇'
)

demo1.launch()

# Running on public URL: https://9e04c9aaa71007243a.gradio.live ==> 여기 접속하면 확인가능

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9e04c9aaa71007243a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo1.close() # 안끄면 계속 돌아가므로 끄는 함수

Closing server running on port: 7860


In [ ]:
def echo_bot(message, history):
  return f"Echo: {message}"


demo2 = gr.ChatInterface(
    fn = echo_bot,  # fnction은 입력에 대한 인터페이스가 어떻게 반응하는지
    title = '에코(Echo) 챗봇',

    examples = ['안녕하세요', '오늘 날씨 어때요', "FAQ 챗봇 테스트"]
)

demo2.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c569c0c6adec5137c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo2.close()

Closing server running on port: 7861


In [ ]:
# Gradio에서는 인터페이스 구성의 자유도를 위해 블럭(Blocks)이라는 기능을 사용
# with과 block을 이용해서 아래에 with 구문 블럭을 차곡차곡 쌓는다는 느낌.

with gr.Blocks(title='custom layout') as demo3:
  gr.Markdown('custom layout demo')
  with gr.Row():
    with gr.Column(scale=2):
      input_text = gr.Textbox(label='질문') # 스케일 2
      submit_btn = gr.Button('전송') # 스케일 2
    with gr.Column(scale=1):
      category_output = gr.Textbox(label='카테고리') # 스케일 1
  output_text = gr.Textbox(label = '답변')

  def process(text):
    cat = "기술" if any(kw in text for kw in ['오류', '설치', '연결']) else "일반"
    return cat, f"[{cat}] {text}"

  submit_btn.click(fn=process, inputs=input_text, outputs=[category_output, output_text])

demo3.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38839c60ce3f8bba75.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo3.close()

Closing server running on port: 7861


In [59]:
# Gradio 인터페이스에 LLM 기능 붙이는 단계
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(model = 'gpt-4o-mini')

def chat_with_llm(message, history):
  # 중괄호 {} 대신 대괄호 []를 사용하여 리스트를 만듭니다.
  messages = [SystemMessage(content = "당신은 친절한 한국어 어시스턴트입니다")]

  # for human, ai in history:
  #   messages.append(HumanMessage(content=human))
  #   messages.append(AIMessage(content=ai))

  for mmm in history:
    if mmm['role'] == 'user':
      messages.append(HumanMessage(content=mmm['content']))
    else:
      messages.append(AIMessage(content=mmm['content']))

  messages.append(HumanMessage(content=message))

  return llm.invoke(messages).content

demo = gr.ChatInterface(
    type='messages',
    fn = chat_with_llm,
    title = '챗봇',
    examples = ['안녕하세요', '오늘 날씨 어때요', 'FAQ 챗봇 테스트']
)

demo.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e0dd24164dc034ff9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.close()

Closing server running on port: 7865


In [62]:
faq_data = [
    {"category": "계정",
     "question": "비밀번호를 잊어버렸습니다. 어떻게 초기화하나요?",
     "answer": "IT 포털(it.company.com)에서 '비밀번호 재설정' 버튼을 클릭하세요. 등록된 이메일로 재설정 링크가 발송됩니다."},
    {"category": "계정",
     "question": "계정이 잠겼습니다. 어떻게 해제하나요?",
     "answer": "5회 이상 비밀번호를 틀리면 계정이 잠깁니다. IT 헬프데스크(내선 1234)에 연락하세요."},
    {"category": "계정",
     "question": "신규 계정은 어떻게 만드나요?",
     "answer": "신규 입사자는 인사팀에서 IT팀에 요청합니다. 입사 당일 계정 정보가 이메일로 발송됩니다."},
    {"category": "계정",
     "question": "2단계 인증(MFA)을 설정하려면?",
     "answer": "IT 포털 > 보안 설정 > MFA 활성화에서 설정합니다. Google Authenticator 앱을 사용하세요."},
]

llm = ChatOpenAI(model = 'gpt-4o-mini')

# context를 주입해서 지식베이스(knowledge base)를 만들어준다고 생각하면 좋음
faq_context = '\n'.join([f"Q: {item['question']}\n A:{item['answer']}" for item in faq_data])

In [63]:
faq_context

"Q: 비밀번호를 잊어버렸습니다. 어떻게 초기화하나요?\n A:IT 포털(it.company.com)에서 '비밀번호 재설정' 버튼을 클릭하세요. 등록된 이메일로 재설정 링크가 발송됩니다.\nQ: 계정이 잠겼습니다. 어떻게 해제하나요?\n A:5회 이상 비밀번호를 틀리면 계정이 잠깁니다. IT 헬프데스크(내선 1234)에 연락하세요.\nQ: 신규 계정은 어떻게 만드나요?\n A:신규 입사자는 인사팀에서 IT팀에 요청합니다. 입사 당일 계정 정보가 이메일로 발송됩니다.\nQ: 2단계 인증(MFA)을 설정하려면?\n A:IT 포털 > 보안 설정 > MFA 활성화에서 설정합니다. Google Authenticator 앱을 사용하세요."

In [66]:
prompt = ChatPromptTemplate.from_messages([
    ('system', f"너는 사내 IT 지원팀 챗봇이야. 아래 제공된 FAQ 데이터를 바탕으로 사용자 질문에 친절하게 답해줘"
    f"데이터에 없는 내용은 'IT 헬프데스크(1234번)으로 문의해주세요' 라고 안내해줘\n\n[FAQ데이터]\n{faq_context}"
    ),
    ('human', '{question}')
])

chain = prompt | llm | StrOutputParser()

def chat_response(message, history):
  response = chain.invoke({'question': message})
  return response

demo = gr.ChatInterface(
    #type='messages',
    fn = chat_response,
    title = '사내지원챗봇',
    examples = ['비밀번호 어떻게 초기화하나요', 'Wi-fi가 너무 느려요', '2단계 인증 어떻게 설정하나요'],
    description = '무엇을 도와드릴까요?'
  )

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14e14381c42c1c3c38.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
